# Overall Pipeline & Orchestration
This notebook demonstrates the complete `stats-transformer` pipeline end-to-end, showing both inline usage of components and full external YAML orchestration as seen in the README.

## 1. Using Components with Inline Configuration
First, we show how you might use individual components like the `FeatureEngineer` directly with an inline dictionary.

In [1]:
import pandas as pd
import yaml
from stats_transformer.data import load_sample_data
from stats_transformer.featurization.feature_engineering import FeatureEngineer

df = load_sample_data()

inline_config = """
featurization:
  entity_column: country
  date_column: date
  period: annual
  transformations:
    - lag1
    - changepct
  transformation_data:
    - gdp
    - inflation
"""

config_dict = yaml.safe_load(inline_config)
fe_config = config_dict['featurization']

engineer = FeatureEngineer(
    params_path=None,
    transformations=fe_config.get('transformations'),
    entity_column=fe_config.get('entity_column'),
    date_column=fe_config.get('date_column'),
    period=fe_config.get('period'),
    data_columns=fe_config.get('transformation_data'),
    verbose=True
)

df_features = engineer.fit_transform(df)
print('\nFeaturized Data Shape:', df_features.shape)
display(df_features.head())


Featurized Data Shape: (11490, 8)


,date,inflation,gdp,country,gdp_changepct,gdp_lag1,inflation_changepct,inflation_lag1
0,1985,4.032258,68.82939,ABW,NaN,NaN,NaN,NaN
1,1986,1.073966,68.82134,ABW,-0.000117,68.82939,-0.733656,4.032258
2,1987,3.639953,68.89938,ABW,0.001134,68.82134,2.389261,1.073966
3,1988,3.121032,68.76744,ABW,-0.001915,68.89938,-0.142562,3.639953
4,1989,3.989460,68.79722,ABW,0.000433,68.76744,0.278250,3.121032


## 2. Using the Orchestrator with an External YAML
We will write a `params.yaml` file to disk. This is the recommended configuration-driven approach.

In [2]:
import os
from importlib.resources import files

# Get path to the sample dataset to feed into the pipeline yaml
data_path = str(files("stats_transformer.data").joinpath("macrodb_gdp_inflation.parquet"))

yaml_content = f"""\
data:
  datasets:
    - name: macro_data
      path: {data_path}
      frequency: annual
      source_period: annual
      resample:
        target_period: annual
  merge:
    on: [country, date]
    how: outer
    output_path: data/pipeline/resampled_merged.parquet
  featurization:
    entity_column: country
    date_column: date
    period: annual
    transformations:
      - lag1
      - changepct
    transformation_data:
      - gdp_Y
      - inflation_Y
    output_path: data/pipeline/features.csv

model:
  model_type: robust_ols
  target_variable: gdp_Y
  independent_variables:
    - inflation_Y
  summary_output_path: reports/model_summary.json

visualization:
  output_dir: reports/visualizations
  regression:
    plots: [coefficient_plot]
"""

# os.makedirs('references/configs', exist_ok=True)
# with open('references/configs/pipeline.yaml', 'w') as f:
#     f.write(yaml_content)
# print('Created references/configs/pipeline.yaml')

## 3. Running the Pipeline
Now we instantiate the `Pipeline` class pointing to our YAML and run it stage by stage.

In [3]:
from stats_transformer.pipeline import Pipeline

pipeline = Pipeline(params_path="references/configs/pipeline.yaml")

# Run the resampling and merging stage
print("--- Running Resample Stage ---")
merged_data = pipeline.run(stage="resample")

# Run the featurization stage
print("\n--- Running Features Stage ---")
transformed_data = pipeline.run(stage="features")

# Run the modeling stage
print("\n--- Running Regression Stage ---")
model_results = pipeline.run(stage="regression")

# Generate and save visualizations
print("\n--- Running Visualization Stage ---")
pipeline.run(stage="visualization")

print("\nPipeline execution complete. Check the 'reports/visualizations' directory for output images.")

--- Running Resample Stage ---

--- Running Features Stage ---

--- Running Regression Stage ---

--- Running Visualization Stage ---
2026-05-08 10:38:32,443 - INFO - Loaded parameters from references/configs/pipeline.yaml
2026-05-08 10:38:32,444 - INFO - Creating coefficient plot from JSON
2026-05-08 10:38:32,523 - INFO - Saved visualization to reports/visualizations/regression/coefficient_plot.png
2026-05-08 10:38:32,524 - INFO - Loaded parameters from references/configs/pipeline.yaml


/Users/cory/Desktop/stats-transformer/src/stats_transformer/visualization/eda/data_viz.py:65: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


2026-05-08 10:38:37,057 - INFO - Saved visualization to reports/visualizations/time_series/time_series.png

Pipeline execution complete. Check the 'reports/visualizations' directory for output images.
